For a comprehensive introduction to the theoretical setup of learning from ordinal data, a key resource is the focus article by [Qiao (2015)](https://wires.onlinelibrary.wiley.com/doi/abs/10.1002/wics.1357), which discusses the fundamental trade-off between model flexibility and interpretability while emphasizes the importance of choosing an appropriate loss function for ordinal problems. 

For broader statistical learning foundations, the seminal books by Hastie, Tibshirani, and Friedman (2009), "[The Elements of Statistical Learning](https://www.sas.upenn.edu/~fdiebold/NoHesitations/BookAdvanced.pdf)", and Duda, Hart, and Stork (2001), "[Pattern Classification](https://github.com/junliu-cn/Books/blob/master/pattern%20classification%20duda.pdf)", are essential references 

### __Loss Functions for Ordinal Data__

The following table summarizes some of the most relevant loss functions, their core mechanisms, and recent references.

<table>
    <tr>
        <td>Loss Function</td>
        <td>Core Strategy</td>
        <td>Key Publications & References</td>
    </tr>
    <tr>
        <td>Soft Labeling Losses</td>
        <td>Replaces one-hot encoded labels with "soft" labels (e.g., from a probability distribution) to encode class proximity. The loss (often cross-entropy) then measures distance to this soft target.</td>
        <td> <code>dlordinal</code> package  implements Binomial, Poisson, Exponential, Beta, and Triangular losses. SLACE (Soft Labels Accumulating Cross Entropy) , introduced by Nachmani et al. (2025) in Proceedings of the AAAI Conference on Artificial Intelligence, ensures monotonicity and balance sensitivity .</td>
    </tr>
    <tr>
        <td>Regularized Cross-Entropy</td>
        <td>Adds a penalty term to the standard cross-entropy loss to enforce a unimodal distribution of predicted probabilities, ensuring higher probabilities for classes near the true label.</td>
        <td>CO and CO2 (with a margin) by Albuquerque et al. (2022) in Mathematics . The Quasi-Unimodal Loss (QUL) , also from Albuquerque et al. (2022), relaxes the unimodal constraint by focusing on the neighborhood around the true class .</td>
    </tr>
    <tr>
        <td>Threshold-Based & Rank-Consistent Losses</td>
        <td>Based on the principle that for ordinal data, the cumulative probability should increase monotonically. These losses often use multiple thresholds to divide the output space.</td>
        <td>Cumulative Link Models (CLM) , implemented as an output layer in the dlordinal package . Ordinal Hyperplane Loss (OHPL) , proposed by Vanderheyden and Xie (2018) in the IEEE International Conference on Big Data, learns a space where class order is maintained by distance to class centroids .</td>
    </tr>
    <tr>
        <td>Metric-Based & Decomposition Losses</td>
        <td>Transforms the ordinal problem into multiple binary sub-problems and uses a loss function that accounts for errors across all of them, considering the ordinal scale.</td>
        <td>Weighted Kappa (WK) Loss , based on a continuous version of the Quadratic Weighted Kappa metric, is implemented in dlordinal . Ordinal Binary Decomposition (OBD) with ECOC distance loss, also in dlordinal, uses Error-Correcting Output Codes to maintain ordinal relationships .</td>
    </tr>
</table>
	
#### __How to Choose and Implement a Loss Function__

Selecting the right loss function depends on your specific data and goals:

1. For tabular data, newer functions like SLACE have shown state-of-the-art performance .

2. For computer vision tasks (e.g., age estimation, medical image grading), Soft Labeling and Regularized Cross-Entropy losses are popular and effective.

3. The OHPL loss is designed to scale well for large datasets with ordinal classes .

To facilitate experimentation, the <code>dlordinal</code> Python package (Bérchez-Moreno et al., 2024) provides a unified, PyTorch-based implementation of many state-of-the-art ordinal loss functions, output layers, and evaluation metrics, making it a practical starting point for applying these methods .


#### __What about SoftMax for ordinal data__

##### __1. Ordinal Softmax (CORAL Framework)__
The ordinal softmax activation function is specifically designed for [ordinal regression](https://www.sciencedirect.com/science/article/pii/S016786552030413X) and is implemented in the [coral-ordinal Python package](https://github.com/haoyang27/coral-ordinal).

How it works:

 * Instead of a single softmax over K classes, it uses K-1 binary tasks

 * The model learns rank logits that are monotonically increasing (ensuring order)

 * The ordinal softmax transforms these logits into probabilities for each category


##### __2. Softmax with Fixed Linear Combinations__
[Beckham and Pal (2016)](https://arxiv.org/abs/1612.00775) proposed a simple but effective approach: use a standard softmax hidden layer but compute the final prediction as $a^{T}f(x)$, where a = [0, 1, 2, ..., K-1] is a fixed weight vector. Squared error $(c - a^{T}f(x))^{2}$ instead of cross-entropy .

##### __3. Softmax GARCH for Time Series__
For ordinal time series data, researchers have developed softmax GARCH models that use the softmax function as a response function for multinomial conditional distributions. [Neural Softmax GARCH](https://link.springer.com/article/10.1007/s00477-023-02591-1): Combines softmax response with artificial neural networks for flexible modeling of ordinal sequences.

##### __4. Harville/Henery Softmax Models__
The [ohenery R package](https://rdrr.io/cran/ohenery/man/ohenery.html) implements softmax regression for ordinal outcomes under the Harville and Henery models. Application: Originally designed for ranking problems (e.g., horse races, competitions) where the order of finishers matters. Formula example (Harville model): $\mathbb{P}(\text{contestant 11 wins}) = \mu_{11} / \Sigma \mu_{i}$, where $\mu_{i} = \exp(x_{i}^{T}\beta)$

<table>
    <tr>
        <td>Method</td>
        <td>Core Idea</td>
        <td>Best For</td>
        <td>Key Reference</td>
    </tr>
    <tr>
        <td>Ordinal Softmax</td>
        <td>Monotonic rank logits</td>
        <td>General ordinal classification</td>
        <td>Cao et al. (2019)</td>
    </tr>
    <tr>
        <td>Fixed Linear Softmax</td>
        <td>$a^{T}f(x)$ with $a = [0,...,K-1]$</td>
        <td>Medical image grading</td>
        <td>Beckham & Pal (2016)</td>
    </tr>
    <tr>
        <td>Softmax GARCH</td>
        <td>Softmax response with time series dependence</td>
        <td>Environmental/economic time series</td>
        <td>Jahn (2023)</td>
    </tr>
    <tr>
        <td>Harville/Henery</td>
        <td>Recursive softmax for rankings</td>
        <td>Competition/race outcome prediction</td>
        <td>Harville (1973), Henery (1981)</td>
    </tr>
</table>


In [ ]:
"""
Comparative Example: CORAL vs Ordinal Softmax vs SLACE for Ordinal Regression
Dataset: Wine Quality Dataset (ordinal wine ratings from 0-10)
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# ============================================================================
# 1. CREATE SYNTHETIC ORDINAL DATASET (since wine dataset isn't ordinal)
# ============================================================================

def create_ordinal_dataset(n_samples=1000, n_features=10, n_classes=5):
    """
    Create synthetic ordinal dataset with monotonic relationships
    """
    X = np.random.randn(n_samples, n_features)
    
    # Create ordinal labels with monotonic relationship to features
    # Higher values in first few features lead to higher classes
    linear_pred = 0.5 * X[:, 0] + 0.3 * X[:, 1] - 0.2 * X[:, 2] + 0.1 * X[:, 3]
    
    # Add noise
    linear_pred += 0.1 * np.random.randn(n_samples)
    
    # Define thresholds for ordinal classes
    thresholds = np.percentile(linear_pred, [20, 40, 60, 80])
    
    # Create ordinal labels
    y = np.zeros(n_samples, dtype=int)
    for i, thresh in enumerate(thresholds):
        y[linear_pred > thresh] = i + 1
    
    return X, y

# Generate dataset
X, y = create_ordinal_dataset(n_samples=2000, n_features=15, n_classes=5)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

# Create data loaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
print(f"Class distribution: {np.bincount(y_train)}")

# ============================================================================
# 2. BASE NEURAL NETWORK ARCHITECTURE
# ============================================================================

class BaseModel(nn.Module):
    """Base neural network for feature extraction"""
    def __init__(self, input_dim, hidden_dims=[64, 32]):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3)
            ])
            prev_dim = hidden_dim
        
        self.features = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.features(x)

# ============================================================================
# 3. CORAL IMPLEMENTATION
# ============================================================================
# Based on: Cao et al. (2019) "Rank-consistent ordinal regression for neural networks"

class CoralModel(nn.Module):
    """CORAL: Consistent Rank Logits for Ordinal Regression"""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.num_ordinal_targets = num_classes - 1
        
        # Feature extractor
        self.base = BaseModel(input_dim)
        
        # Coral layer - produces num_classes-1 logits that should be monotonic
        self.coral_layer = nn.Linear(32, self.num_ordinal_targets)
        
        # Ensure monotonicity by using cumulative probabilities
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        features = self.base(x)
        logits = self.coral_layer(features)  # Shape: (batch_size, num_classes-1)
        
        # Apply sigmoid to get probabilities P(y > k)
        p_greater = self.sigmoid(logits)
        
        # Convert to class probabilities using cumulative differences
        # P(y = 0) = 1 - P(y > 0)
        # P(y = k) = P(y > k-1) - P(y > k) for 1 <= k <= K-2
        # P(y = K-1) = P(y > K-2)
        
        batch_size = x.size(0)
        class_probs = torch.zeros(batch_size, self.num_classes, device=x.device)
        
        # First class
        class_probs[:, 0] = 1 - p_greater[:, 0]
        
        # Middle classes
        for k in range(1, self.num_classes - 1):
            class_probs[:, k] = p_greater[:, k-1] - p_greater[:, k]
        
        # Last class
        class_probs[:, -1] = p_greater[:, -1]
        
        # Ensure non-negative and sum to 1
        class_probs = torch.clamp(class_probs, min=0)
        class_probs = class_probs / class_probs.sum(dim=1, keepdim=True)
        
        return class_probs, p_greater
    
    def coral_loss(self, predictions, targets):
        """
        Coral loss: binary cross-entropy on each ordinal target
        """
        _, p_greater = self.forward(predictions)
        
        # Convert targets to binary labels for each ordinal target
        # For target t, the binary label for task k is 1 if t > k else 0
        targets_expanded = targets.unsqueeze(1).expand(-1, self.num_ordinal_targets)
        thresholds = torch.arange(self.num_ordinal_targets, device=targets.device).unsqueeze(0)
        binary_targets = (targets_expanded > thresholds).float()
        
        # Binary cross-entropy loss for each ordinal task
        loss_fn = nn.BCEWithLogitsLoss()
        loss = loss_fn(p_greater, binary_targets)
        
        return loss

# ============================================================================
# 4. ORDINAL SOFTMAX IMPLEMENTATION
# ============================================================================
# Based on: Traditional ordinal softmax with monotonic constraints

class OrdinalSoftmaxModel(nn.Module):
    """Ordinal Softmax with learned thresholds"""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.num_classes = num_classes
        
        # Feature extractor
        self.base = BaseModel(input_dim)
        
        # Final layer for logits
        self.fc = nn.Linear(32, num_classes)
        
    def forward(self, x):
        features = self.base(x)
        logits = self.fc(features)
        
        # Standard softmax (ignores order)
        probs = torch.softmax(logits, dim=1)
        
        return probs, logits
    
    def loss(self, probs, targets):
        """
        Standard cross-entropy loss
        """
        loss_fn = nn.CrossEntropyLoss()
        return loss_fn(probs, targets)

# ============================================================================
# 5. SLACE IMPLEMENTATION (Simplified)
# ============================================================================
# Based on: Nachmani et al. (2025) "SLACE: Soft Labels Accumulating Cross Entropy"

class SLACEModel(nn.Module):
    """
    Simplified SLACE: Soft Labels Accumulating Cross Entropy
    Core idea: Accumulate probabilities across ordinal categories and
    create soft labels based on ordinal distance
    """
    def __init__(self, input_dim, num_classes, temperature=2.0, sigma=1.0):
        super().__init__()
        self.num_classes = num_classes
        self.temperature = temperature  # For soft label smoothing
        self.sigma = sigma  # Controls width of soft labels
        
        # Feature extractor
        self.base = BaseModel(input_dim)
        
        # Final layer
        self.fc = nn.Linear(32, num_classes)
        
    def create_soft_labels(self, targets):
        """
        Create soft labels based on ordinal distance
        Uses Gaussian smoothing around true class
        """
        batch_size = targets.size(0)
        soft_labels = torch.zeros(batch_size, self.num_classes, device=targets.device)
        
        # Create ordinal distance matrix
        class_indices = torch.arange(self.num_classes, device=targets.device).float()
        targets_float = targets.float().unsqueeze(1)
        
        # Gaussian-based soft labels
        distances = torch.abs(class_indices.unsqueeze(0) - targets_float)
        soft_labels = torch.exp(-(distances ** 2) / (2 * self.sigma ** 2))
        
        # Apply temperature for label smoothing
        soft_labels = soft_labels ** (1.0 / self.temperature)
        
        # Normalize to sum to 1
        soft_labels = soft_labels / soft_labels.sum(dim=1, keepdim=True)
        
        return soft_labels
    
    def forward(self, x):
        features = self.base(x)
        logits = self.fc(features)
        probs = torch.softmax(logits, dim=1)
        return probs, logits
    
    def slace_loss(self, logits, targets):
        """
        SLACE loss: Cross-entropy with soft labels and ordinal accumulation
        """
        # Create soft labels based on ordinal distance
        soft_labels = self.create_soft_labels(targets)
        
        # Accumulate probabilities across ordinal categories
        # This is the key idea: P(y >= k) should be monotonic
        probs = torch.softmax(logits, dim=1)
        cum_probs = torch.cumsum(probs, dim=1)  # Accumulated probabilities
        
        # Penalize non-monotonicity in accumulated probabilities
        # This is a simplified version of the SLACE constraint
        log_probs = torch.log_softmax(logits, dim=1)
        
        # Cross-entropy with soft labels
        loss = -torch.sum(soft_labels * log_probs, dim=1).mean()
        
        # Add monotonicity penalty
        monotonicity_penalty = torch.mean(torch.relu(cum_probs[:, 1:] - cum_probs[:, :-1]))
        
        return loss + 0.1 * monotonicity_penalty

# ============================================================================
# 6. TRAINING FUNCTION
# ============================================================================

def train_model(model, train_loader, test_loader, model_type, epochs=50):
    """
    Train a model and track metrics
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    if model_type == 'coral':
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    else:
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=10)
    
    train_losses = []
    test_accuracies = []
    test_maes = []
    
    for epoch in tqdm(range(epochs), desc=f"Training {model_type}"):
        # Training
        model.train()
        epoch_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            
            if model_type == 'coral':
                probs, _ = model(batch_X)
                loss = model.coral_loss(batch_X, batch_y)
            elif model_type == 'slace':
                probs, logits = model(batch_X)
                loss = model.slace_loss(logits, batch_y)
            else:  # ordinal_softmax
                probs, logits = model(batch_X)
                loss = model.loss(logits, batch_y)
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        
        # Evaluation
        model.eval()
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                
                if model_type == 'coral':
                    probs, _ = model(batch_X)
                    preds = torch.argmax(probs, dim=1)
                else:
                    probs, _ = model(batch_X)
                    preds = torch.argmax(probs, dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(batch_y.numpy())
        
        accuracy = accuracy_score(all_targets, all_preds)
        mae = mean_absolute_error(all_targets, all_preds)
        test_accuracies.append(accuracy)
        test_maes.append(mae)
        
        scheduler.step(avg_loss)
    
    return {
        'train_losses': train_losses,
        'test_accuracies': test_accuracies,
        'test_maes': test_maes,
        'final_accuracy': test_accuracies[-1],
        'final_mae': test_maes[-1],
        'final_predictions': all_preds,
        'true_labels': all_targets
    }

# ============================================================================
# 7. TRAIN AND COMPARE MODELS
# ============================================================================

print("\n" + "="*60)
print("TRAINING AND COMPARING MODELS")
print("="*60)

# Initialize models
input_dim = X_train.shape[1]
num_classes = len(np.unique(y))

coral_model = CoralModel(input_dim, num_classes)
ordinal_softmax_model = OrdinalSoftmaxModel(input_dim, num_classes)
slace_model = SLACEModel(input_dim, num_classes, temperature=2.0, sigma=1.0)

# Train models
coral_results = train_model(coral_model, train_loader, test_loader, 'coral', epochs=50)
ordinal_results = train_model(ordinal_softmax_model, train_loader, test_loader, 'ordinal_softmax', epochs=50)
slace_results = train_model(slace_model, train_loader, test_loader, 'slace', epochs=50)

# ============================================================================
# 8. RESULTS COMPARISON
# ============================================================================

print("\n" + "="*60)
print("FINAL RESULTS COMPARISON")
print("="*60)

results_df = pd.DataFrame({
    'Model': ['CORAL', 'Ordinal Softmax', 'SLACE (Simplified)'],
    'Test Accuracy': [
        coral_results['final_accuracy'],
        ordinal_results['final_accuracy'],
        slace_results['final_accuracy']
    ],
    'Test MAE': [
        coral_results['final_mae'],
        ordinal_results['final_mae'],
        slace_results['final_mae']
    ]
})

print(results_df.to_string(index=False))

# ============================================================================
# 9. VISUALIZATION
# ============================================================================

# Create visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Training loss curves
axes[0, 0].plot(coral_results['train_losses'], label='CORAL', linewidth=2)
axes[0, 0].plot(ordinal_results['train_losses'], label='Ordinal Softmax', linewidth=2)
axes[0, 0].plot(slace_results['train_losses'], label='SLACE', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Training Loss')
axes[0, 0].set_title('Training Loss Curves')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Test accuracy curves
axes[0, 1].plot(coral_results['test_accuracies'], label='CORAL', linewidth=2)
axes[0, 1].plot(ordinal_results['test_accuracies'], label='Ordinal Softmax', linewidth=2)
axes[0, 1].plot(slace_results['test_accuracies'], label='SLACE', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Test Accuracy')
axes[0, 1].set_title('Test Accuracy Over Time')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Test MAE curves
axes[0, 2].plot(coral_results['test_maes'], label='CORAL', linewidth=2)
axes[0, 2].plot(ordinal_results['test_maes'], label='Ordinal Softmax', linewidth=2)
axes[0, 2].plot(slace_results['test_maes'], label='SLACE', linewidth=2)
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Test MAE')
axes[0, 2].set_title('Test MAE Over Time')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Confusion matrices
models = ['CORAL', 'Ordinal Softmax', 'SLACE']
results = [coral_results, ordinal_results, slace_results]

for i, (name, res) in enumerate(zip(models, results)):
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(res['true_labels'], res['final_predictions'])
    
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[1, i], cmap='Blues')
    axes[1, i].set_xlabel('Predicted')
    axes[1, i].set_ylabel('True')
    axes[1, i].set_title(f'{name} Confusion Matrix')

plt.tight_layout()
plt.savefig('ordinal_regression_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================================
# 10. ADDITIONAL ANALYSIS: MONOTONICITY CHECK
# ============================================================================

def check_monotonicity(model, X_test_tensor, device):
    """
    Check if predictions respect ordinal monotonicity
    """
    model.eval()
    with torch.no_grad():
        X_test_tensor = X_test_tensor.to(device)
        
        if isinstance(model, CoralModel):
            probs, p_greater = model(X_test_tensor)
            # Check if p_greater is monotonic decreasing
            monotonic = all(torch.all(p_greater[:, i] >= p_greater[:, i+1]) 
                          for i in range(p_greater.size(1)-1))
            return monotonic, p_greater.cpu().numpy()
        else:
            probs, _ = model(X_test_tensor)
            # For non-CORAL, check if cumulative probs are monotonic
            cum_probs = torch.cumsum(probs, dim=1)
            monotonic = all(torch.all(cum_probs[:, i] <= cum_probs[:, i+1]) 
                          for i in range(cum_probs.size(1)-1))
            return monotonic, cum_probs.cpu().numpy()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
coral_monotonic, coral_cum = check_monotonicity(coral_model, X_test_tensor[:100], device)
softmax_monotonic, softmax_cum = check_monotonicity(ordinal_softmax_model, X_test_tensor[:100], device)
slace_monotonic, slace_cum = check_monotonicity(slace_model, X_test_tensor[:100], device)

print("\n" + "="*60)
print("MONOTONICITY CHECK (First 100 test samples)")
print("="*60)
print(f"CORAL maintains monotonicity: {coral_monotonic}")
print(f"Ordinal Softmax maintains monotonicity: {softmax_monotonic}")
print(f"SLACE maintains monotonicity: {slace_monotonic}")

# ============================================================================
# 11. SUMMARY STATISTICS
# ============================================================================

print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

for model_name, results in zip(['CORAL', 'Ordinal Softmax', 'SLACE'], 
                               [coral_results, ordinal_results, slace_results]):
    print(f"\n{model_name}:")
    print(f"  Final Accuracy: {results['final_accuracy']:.4f}")
    print(f"  Final MAE: {results['final_mae']:.4f}")
    print(f"  Best Accuracy: {max(results['test_accuracies']):.4f}")
    print(f"  Best MAE: {min(results['test_maes']):.4f}")

 Theoretical Differences
Aspect	CORAL	Ordinal Softmax	SLACE
Core Mechanism	K-1 binary tasks with monotonic constraints	Single softmax over all classes	Soft labels + accumulated probabilities
Monotonicity	Explicitly enforced	Not enforced	Encouraged via penalty
Loss Function	Binary cross-entropy on ordinal tasks	Standard cross-entropy	Soft CE + monotonicity penalty
Output	Class probabilities via cumulative differences	Direct class probabilities	Class probabilities
2. Expected Performance Characteristics
CORAL: Best at maintaining ordinal relationships; tends to make "safer" predictions (predicting closer to true class when uncertain)

Ordinal Softmax: Faster training; may ignore ordinal information; can produce non-monotonic predictions

SLACE: Balanced approach; soft labels help with generalization; monotonicity penalty encourages order

3. When to Use Each
Scenario	Recommended Model	Reason
Medical grading (e.g., disease severity)	CORAL	Critical to maintain ordinal relationships
Large-scale classification	Ordinal Softmax	Faster training, simpler implementation
Noisy labels	SLACE	Soft labels help handle annotation noise
Small datasets	CORAL	Inductive bias helps prevent overfitting
Time series ordinal data	SLACE	Accumulation property useful for temporal data
4. Practical Considerations


In [ ]:
# 1. Weighted CORAL for imbalanced data
class WeightedCoralModel(CoralModel):
    def coral_loss(self, predictions, targets):
        _, p_greater = self.forward(predictions)
        
        # Compute class weights
        class_counts = torch.bincount(targets)
        weights = 1.0 / class_counts.float()
        weights = weights / weights.sum()
        
        # Apply weights to loss
        targets_expanded = targets.unsqueeze(1).expand(-1, self.num_ordinal_targets)
        thresholds = torch.arange(self.num_ordinal_targets, device=targets.device).unsqueeze(0)
        binary_targets = (targets_expanded > thresholds).float()
        
        loss_fn = nn.BCEWithLogitsLoss(weight=weights[targets].unsqueeze(1))
        return loss_fn(p_greater, binary_targets)

# 2. Ensemble of ordinal models
class OrdinalEnsemble:
    def __init__(self, models, weights=None):
        self.models = models
        self.weights = weights if weights else [1/len(models)] * len(models)
    
    def predict(self, X):
        predictions = []
        for model in self.models:
            model.eval()
            with torch.no_grad():
                probs, _ = model(X)
                predictions.append(probs.numpy())
        
        # Weighted average
        ensemble_probs = np.average(predictions, axis=0, weights=self.weights)
        return np.argmax(ensemble_probs, axis=1)

CORAL: Cao, W., Mirjalili, V., & Raschka, S. (2019). Rank-consistent ordinal regression for neural networks.

SLACE: Nachmani, E., et al. (2025). SLACE: Soft Labels Accumulating Cross Entropy. AAAI Conference on Artificial Intelligence.

Ordinal Softmax: Beckham, C., & Pal, C. (2016). A simple baseline for ordinal regression.